# 광고 도메인 라벨링
광고명 기준 도메인 분류 흐름 보존
룰 기반 분류 후 Gemini 검증, 최종본은 재호출 없이 검증

## 설정
라이브러리와 경로 확인

In [1]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import csv
import hashlib
import json
import math
import re
import threading
import time
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google import genai
except Exception:
    genai = None

pd.set_option('display.max_columns', 100)

## 설정과 원본 로드
원본 광고목록 445,260행 기준
Gemini 호출은 완료본 확보 후 비활성


In [2]:
# 실행 옵션
# Gemini 기반 광고명 분류 모델
# 완료 라벨 확보 후 재호출 방지
RUN_GEMINI = False
WRITE_FINAL_CSV = False
MODEL_ID = 'gemini-2.0-flash'
GEMINI_JSON_PATH = '개인 Gemini API 키 JSON 경로'

# 원본 로드
df_raw = pd.read_csv('C:/Users/ckcma/Downloads/temp2/IVE_광고목록.csv', encoding='utf-8-sig', low_memory=False)

# 분석 기간
analysis_sdate = pd.Timestamp('2025-07-26')
analysis_edate = pd.Timestamp('2025-08-25')
future_fix_date = pd.Timestamp('2025-08-29')

df_raw.shape


(445260, 30)

## Gemini 호출 흔적
Gemini API 비용·시간·결과 변동성 존재
최종 분류본 기준 검증, 재호출 로직만 보존


In [3]:
# API 호출 구조 보존
#
# from google import genai
# GEMINI_JSON_PATH = Path('개인 API 키 JSON 경로')
# MODEL_ID = 'gemini-2.0-flash'
# RUN_MODE = 'prod'       # test/prod 구분
# TEST_N = 1000           # 테스트 실행 시 샘플 수
# BATCH_SIZE = 40         # Gemini 한 번 호출에 넣던 광고명 개수
# GLOBAL_TARGET_RPM = 15  # 분당 호출 수 제한
# MAX_RETRIES = 6         # 실패 시 재시도 횟수
# MAX_BACKOFF_SEC = 120   # 재시도 대기 최대 초
# COOLDOWN_429_SEC = 30   # 429 제한 발생 시 대기 초
# MIN_SPLIT_SIZE = 6      # 실패한 배치를 더 쪼갤 때 최소 크기
#
# with open(GEMINI_JSON_PATH, 'r', encoding='utf-8') as f:
#     config = json.load(f)
# client = genai.Client(api_key=config['GEMINI_API_KEY'])
# response = client.models.generate_content(model=MODEL_ID, contents=prompt)

if RUN_GEMINI:
    raise RuntimeError('최종 분류본 사용, Gemini 재호출 비활성')

client = None
'Gemini 재호출 제외, 최종 분류본 사용'

# Gemini 응답 수신 함수
# 현재 재호출 방지용 안전장치
def make_answer(question: str) -> str:
    if client is None:
        raise RuntimeError('Gemini API 재호출 제외')
    response = client.models.generate_content(model=MODEL_ID, contents=question)
    return response.text

## 키워드 1차 분류
광고명에 명확한 도메인 키워드가 있는 경우 선분류
모호한 광고명만 Gemini 검증 대상으로 분리

In [4]:
KEYWORDS = {
    1: ['게임','웹툰','드라마','영화','콘텐츠','영상','스트리밍','유튜브','틱톡','sns','소셜','음악','tv','티비'],
    2: ['금융','은행','증권','투자','재테크','대출','캐피탈','카드','보험','적금','예금','포인트','적립','캐시백','리워드','페이','송금','결제','코인','거래소'],
    3: ['헬스','운동','다이어트','건강','통신','요금','알뜰','소개팅','매칭','유틸','예약','교통','지도','서비스','배달','택시','채팅','랜덤채팅','미팅','마사지'],
    4: ['쇼핑','리테일','이커머스','커머스','세일','할인','쿠폰','딜','특가','구매','주문','배송','마켓','스토어','몰','아울렛','멤버십','11번가','쿠팡','네이버쇼핑','ssg','롯데','홈플러스','lf몰','29cm'],
    5: ['test','테스트','tracking','트래킹','qa','dummy','더미','tag','태그','campaign','캠페인','unknown','정체불명','extractor','debug'],
}

domain_name = {1: '엔터', 2: '금융', 3: '라이프', 4: '커머스', 5: '기타'}

# 광고명 비교용 정규화
# 대소문자·공백 차이 제거
# 키워드 hit 계산 전 기준 통일
def normalize_text(x: str) -> str:
    if x is None:
        return ''
    return re.sub(r'\s+', ' ', str(x).strip().lower())

# 도메인 키워드 hit 수 계산
# 여러 도메인 동시 매칭 여부 확인
def rule_score_counts(name: str) -> Dict[int, int]:
    text = normalize_text(name)
    if not text:
        return {}
    return {
        label: sum(1 for kw in kws if kw.lower() in text)
        for label, kws in KEYWORDS.items()
        if any(kw.lower() in text for kw in kws)
    }

# 키워드 단일 매칭 시 라벨 확정
# 모호 케이스 LLM 후보 유지
def rule_classify_num_tie_to_llm(name: str) -> Optional[int]:
    text = normalize_text(name)
    if not text:
        return 5
    scores = rule_score_counts(name)
    if not scores:
        return None
    max_count = max(scores.values())
    winners = [label for label, count in scores.items() if count == max_count]
    return winners[0] if len(winners) == 1 else None

# 룰 기반 선분류
# 미분류 광고명 Gemini 후보 분리
def apply_rule_filter(unique_names: List[str], rule_enabled: bool = True):
    name_to_label: Dict[str, int] = {}
    llm_targets: List[str] = []
    for name in unique_names:
        name = '' if pd.isna(name) else str(name)
        label = rule_classify_num_tie_to_llm(name) if rule_enabled else None
        if label is None:
            llm_targets.append(name)
        else:
            name_to_label[name] = int(label)
    return name_to_label, llm_targets

keyword_cnt = pd.Series({domain_name[k]: len(v) for k, v in KEYWORDS.items()}, name='keyword_count')
keyword_cnt

엔터     14
금융     20
라이프    20
커머스    25
기타     15
Name: keyword_count, dtype: int64

## Gemini 프롬프트
광고명을 짧게 정리해 번호형 프롬프트 구성
출력은 광고명별 domain_label 매핑 형태

In [5]:
# 프롬프트용 광고명 정리
def sanitize_name_for_prompt(x: str, max_len: int = 120) -> str:
    x = str(x).replace('\n', ' ').replace('\r', ' ').strip()
    return x[:max_len] + '...' if len(x) > max_len else x

# 1차 분류 프롬프트
def build_prompt_simple(names: List[str]) -> str:
    joined = '\n'.join(f'{i}: "{sanitize_name_for_prompt(n)}"' for i, n in enumerate(names))
    return f'''
광고명(ads_name) 텍스트 기준 도메인 분류.
아래 숫자 라벨 중 하나만 선택하세요.

[숫자 라벨 정의]
1: 엔터테인먼트(게임/콘텐츠/영상/소셜)
2: 금융(금융/보험/재테크/대출/적립/결제/송금)
3: 라이프스타일(생활앱/유틸/헬스/소개팅/통신/예약/교통/서비스)
4: 커머스(쇼핑/리테일/세일/이커머스/마켓/스토어/멤버십/할인/쿠폰)
5: 기타(테스트/트래킹용/정체불명/캠페인 태그성 또는 위에 해당 없음)

[출력 규칙]
- 반드시 JSON 배열만 출력
- 배열 길이 = 입력 개수, 원소는 정수 1~5

[입력 목록]
{joined}
'''.strip()

# 불일치/기타 라벨 재검토 프롬프트
def build_prompt_detailed(names: List[str]) -> str:
    joined = '\n'.join(f'{i}: "{sanitize_name_for_prompt(n)}"' for i, n in enumerate(names))
    return f'''
광고명(ads_name) 텍스트 기준 도메인 분류.
아래 숫자 라벨 중 하나만 선택하세요.

[숫자 라벨 정의]
1: 엔터테인먼트 - 게임/웹툰/드라마/영화/OTT/스트리밍/아이돌/팬덤/음악/유튜브/틱톡/SNS
2: 금융 - 은행/증권/투자/재테크/대출/카드/보험/적금/예금/연금/송금/간편결제/페이
3: 라이프스타일 - 헬스/운동/의료/통신/예약/교통/택시/배달/숙박/여행/학습/소개팅/유틸
4: 커머스 - 쇼핑/마켓/스토어/특가/딜/쿠폰/할인/장바구니/주문/구매/배송/브랜드
5: 기타 - 테스트/트래킹/태그/캠페인용/정체불명 또는 위 1~4에 근거 부족

[판정 우선순위]
1. 구매/주문/배송/쿠폰/특가 신호 -> 4
2. 대출/보험/금리/계좌/송금 신호 -> 2
3. 게임/웹툰/드라마/팬덤/음악 신호 -> 1
4. 생활 서비스 신호 -> 3
5. 근거 부족/코드성/애매 -> 5

[출력 규칙]
- 반드시 JSON 배열만 출력
- 배열 길이 = 입력 개수, 원소는 정수 1~5

[입력 목록]
{joined}
'''.strip()

'Gemini 프롬프트 준비'

'Gemini 프롬프트 준비'

## API 호출 안전장치
대량 광고명 분류 당시 호출 간격·재시도 로직
429 호출 제한과 일시 오류 대비, 현재 RUN_GEMINI=False


In [6]:
# 호출 간격 제한
class GlobalRateLimiter:
    def __init__(self, min_interval_sec: float):
        self.min_interval_sec = float(min_interval_sec)
        self._lock = threading.Lock()
        self._last_call_ts = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            if self._last_call_ts <= 0:
                self._last_call_ts = now
                return
            elapsed = now - self._last_call_ts
            if elapsed < self.min_interval_sec:
                time.sleep(self.min_interval_sec - elapsed)
            self._last_call_ts = time.time()

GLOBAL_RATE_LIMITER = GlobalRateLimiter(60 / 15)

# Gemini 응답 JSON 배열 추출
def extract_json_array(text: str) -> List[Any]:
    text = (text or '').strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass
    match = re.search(r'\[.*\]', text, flags=re.DOTALL)
    if not match:
        raise ValueError('No JSON array found in model output.')
    obj = json.loads(match.group(0))
    if not isinstance(obj, list):
        raise ValueError('Parsed JSON is not a list.')
    return obj

# 광고명 배치별 Gemini 라벨 수신
# RUN_GEMINI=False
def gemini_call_once(names: List[str], make_answer_fn, prompt_fn) -> Dict[str, int]:
    GLOBAL_RATE_LIMITER.wait()
    raw = make_answer_fn(prompt_fn(names))
    arr = extract_json_array(raw)
    if len(arr) != len(names):
        raise ValueError(f'Batch output size mismatch: got={len(arr)}, expected={len(names)}')
    return {name: int(label) for name, label in zip(names, arr)}

# Gemini 대상 광고명 배치 호출
# 최종 분류본 확보로 실행 제외
def run_batch_classify(names: List[str], make_answer_fn, prompt_fn, batch_size: int = 40) -> Dict[str, int]:
    if not RUN_GEMINI:
        raise RuntimeError('Gemini 호출 제외, 최종 분류본 사용')
    out: Dict[str, int] = {}
    for start in range(0, len(names), batch_size):
        batch = names[start:start + batch_size]
        out.update(gemini_call_once(batch, make_answer_fn, prompt_fn))
    return out

'배치 호출 로직 보존, RUN_GEMINI=False'

'배치 호출 로직 보존, RUN_GEMINI=False'

## 1차 분류 대상
원본 광고명 unique 기준 룰 분류와 Gemini 대상 분리
룰 기반 16,348개, Gemini 대상 147,324개


In [7]:
# 광고명 unique 추출
# 같은 광고명 반복 행은 1번만 분류
unique_names = pd.Series(df_raw['ads_name'].fillna('').astype(str)).drop_duplicates().tolist()

# 룰 분류와 Gemini 후보 분리
# 키워드 단일 매칭은 룰 확정, 모호 케이스는 Gemini 대상
rule_labels, llm_targets = apply_rule_filter(unique_names, rule_enabled=True)

pd.Series({
    'unique_ads_name': len(unique_names),
    'rule_labeled': len(rule_labels),
    'gemini_target': len(llm_targets),
})


unique_ads_name    163672
rule_labeled        16348
gemini_target      147324
dtype: int64

## 불일치 재검토
룰 라벨과 Gemini 라벨 충돌 케이스 재확인 단계
최종 분류본 확보 후 재호출 제외


In [8]:
# 불일치 재검토 흐름 보존
# 완료본 기준 실행 스킵
def reconcile_mismatch_labels(make_answer_fn=None):
    # 룰과 Gemini 결과 충돌 케이스 재검토
    if not RUN_GEMINI:
        return '2차 불일치 재분류 제외, 최종 분류본 사용'

    raise RuntimeError('RUN_GEMINI=True 전환 시 원본 광고명 재분류 로직 연결 필요')

reconcile_mismatch_labels(make_answer)


'2차 불일치 재분류 제외, 최종 분류본 사용'

## 기타 라벨 재검토
기타(label=5) 잔여 광고명 별도 재분류 단계
최종 분류본 확보 후 재호출 제외


In [9]:
# 기타 라벨 재검토 흐름 보존
# 완료본 기준 실행 스킵
def reclassify_label5_only(make_answer_fn=None):
    # 기타 라벨 잔여 광고명 재검토
    if not RUN_GEMINI:
        return '3차 기타 라벨 재분류 제외, 최종 분류본 사용'

    raise RuntimeError('RUN_GEMINI=True 전환 시 기타 라벨 재분류 로직 연결 필요')

reclassify_label5_only(make_answer)


'3차 기타 라벨 재분류 제외, 최종 분류본 사용'

## 최종 라벨 확인
최종 전처리 완료본에서 ads_idx-domain_label 매핑 확인
원본 광고목록에 라벨 1개 컬럼 연결 준비


In [10]:
# 최종 라벨 파일 로드
# ads_idx 기준 원본 연결용 라벨 맵 생성
df_labeled = pd.read_csv(
    'C:/Users/ckcma/Downloads/temp2/04 ADS/광고 목록_도메인 라벨링_전처리완료.csv',
    encoding='utf-8-sig',
    usecols=['ads_idx', 'ads_name', 'domain_label']
)

# 필수 컬럼 확인
need_cols = {'ads_idx', 'ads_name', 'domain_label'}
missing_cols = need_cols - set(df_labeled.columns)
if missing_cols:
    raise ValueError('최종 도메인 라벨 결과 필수 컬럼 누락: ' + str(missing_cols))

# ads_idx 중복 제거 후 라벨 숫자화
label_map = df_labeled[['ads_idx', 'domain_label']].drop_duplicates('ads_idx').copy()
label_map['domain_label'] = pd.to_numeric(label_map['domain_label'], errors='coerce').fillna(5).astype(int)

pd.Series({
    'raw_rows': df_raw.shape[0],
    'raw_cols': df_raw.shape[1],
    'label_ads_idx': label_map['ads_idx'].nunique(),
})


raw_rows         445260
raw_cols             30
label_ads_idx    434740
dtype: int64

## 원본 구조와 결측
라벨 연결 전 원본 컬럼 구조 확인
전처리 대상 결측 컬럼과 결측률 점검


## 라벨 보정
중간 분석 중 광고명 기반 오분류 확인
원스토어 엔터, 상품형 키워드 커머스 수기 보정

In [11]:
# 수기 보정 키워드
# 중간 분석 중 오분류 확인 단어
fix_rules = [
    ('원스토어', 1, '엔터'),
    ('에어매쉬토퍼', 4, '커머스'),
    ('시어서커 패드', 4, '커머스'),
    ('바디필로우', 4, '커머스'),
    ('마사지건', 4, '커머스'),
    ('다이어트', 4, '커머스'),
]

# 원본 광고명 + 기존 라벨 연결
label_work = df_raw[['ads_idx', 'ads_name']].merge(label_map, on='ads_idx', how='left')

# 미매칭 라벨 기타 처리
label_work['domain_label'] = label_work['domain_label'].fillna(5).astype(int)

fix_log = []
total_change = 0
for keyword, target_label, target_name in fix_rules:
    # 광고명 키워드 매칭
    mask = label_work['ads_name'].astype(str).str.contains(keyword, na=False)

    # 기존 라벨과 다른 행만 변경 카운트
    label_diff = label_work['domain_label'] != target_label
    change_mask = np.logical_and(mask, label_diff)
    change_count = int(change_mask.sum())

    # 수기 라벨 반영
    label_work.loc[mask, 'domain_label'] = target_label
    total_change += change_count
    fix_log.append({
        'keyword': keyword,
        'target': target_name,
        'target_label': target_label,
        'matched_rows': int(mask.sum()),
        'changed_rows': change_count,
    })

label_map = label_work[['ads_idx', 'domain_label']].copy()
fix_log = pd.DataFrame(fix_log)
fix_log


,keyword,target,target_label,matched_rows,changed_rows
0,원스토어,엔터,1,623,298
1,에어매쉬토퍼,커머스,4,2,0
2,시어서커 패드,커머스,4,2,0
3,바디필로우,커머스,4,129,0
4,마사지건,커머스,4,12,1
5,다이어트,커머스,4,365,7


In [12]:
# 원본 컬럼 구조
df_raw.info()

# 컬럼별 결측 규모
# 이후 Unknown 보정·컬럼 제거 기준 확인
missing_all = df_raw.isna().sum().reset_index()
missing_all.columns = ['column', 'missing_rows']
missing_all['missing_rate(%)'] = (missing_all['missing_rows'] / len(df_raw) * 100).round(2)

missing_all.sort_values('missing_rows', ascending=False)


<class 'pandas.DataFrame'>
RangeIndex: 445260 entries, 0 to 445259
Data columns (total 30 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   ads_idx             445260 non-null  int64
 1   ads_code            445260 non-null  str  
 2   aff_idx             445260 non-null  int64
 3   adv_idx             445260 non-null  int64
 4   sch_idx             445260 non-null  int64
 5   ads_type            445260 non-null  int64
 6   ads_category        445260 non-null  int64
 7   ads_name            445260 non-null  str  
 8   ads_search          445258 non-null  str  
 9   ads_icon_img        445255 non-null  str  
 10  ads_summary         438199 non-null  str  
 11  ads_guide           444957 non-null  str  
 12  ads_limit           543 non-null     str  
 13  ads_payment         2335 non-null    str  
 14  ads_save_way        444746 non-null  str  
 15  ads_day_cap         445260 non-null  str  
 16  ads_sdate           445260 non-

,column,missing_rows,missing_rate(%)
19,ads_sex_type,445250,100.00
12,ads_limit,444717,99.88
13,ads_payment,442925,99.48
18,ads_package,433998,97.47
10,ads_summary,7061,1.59
14,ads_save_way,514,0.12
11,ads_guide,303,0.07
9,ads_icon_img,5,0.00
8,ads_search,2,0.00
5,ads_type,0,0.00


## 결측·이상치
전처리 전 제거/보정 후보 확인
비용 오류, 미래 종료일, 분석 기간 밖 광고 점검

In [13]:
df_master = df_raw.copy()

# 제거 후보 컬럼
# 이미지/성별/제한/패키지/결제 부가정보
drop_cols = ['ads_icon_img', 'ads_sex_type', 'ads_limit', 'ads_package', 'ads_payment']

# Unknown 대체 컬럼
# 검색어/요약/저장방법/가이드 결측
unknown_cols = ['ads_search', 'ads_summary', 'ads_save_way', 'ads_guide']

# 기간 판단 컬럼
date_cols = ['regdate', 'ads_sdate', 'ads_edate']

# Unknown 대체 전 결측 규모
missing_unknown = df_master[unknown_cols].isna().sum()

# 단가 숫자 변환
contract = pd.to_numeric(df_master['ads_contract_price'], errors='coerce')
reward = pd.to_numeric(df_master['ads_reward_price'], errors='coerce')

# 비용 오류 조건
bad_price = np.logical_or.reduce([
    contract.isna() | reward.isna(),
    contract < reward,
    contract <= 0,
    reward <= 0,
])

# 비용 정상 행 기준 날짜 확인
ads_price = df_master.loc[~bad_price].copy()
date_raw = df_master[date_cols].apply(pd.to_datetime, errors='coerce')
date_price = ads_price[date_cols].apply(pd.to_datetime, errors='coerce')

# 미래 종료일 후보
future_raw = date_raw['ads_edate'].dt.year >= 9999
future_price = date_price['ads_edate'].dt.year >= 9999

# 미래 종료일 기준일 보정
date_price.loc[future_price, 'ads_edate'] = future_fix_date

# 날짜 오류 조건
date_missing = date_price[date_cols].isna().any(axis=1)
start_out = date_price['ads_sdate'] > analysis_edate
end_out = date_price['ads_edate'] < analysis_sdate
date_error = np.logical_or.reduce([
    date_missing,
    start_out,
    end_out,
])

check_before = pd.Series({
    'raw_rows': df_master.shape[0],
    'raw_cols': df_master.shape[1],
    'drop_cols': len(drop_cols),
    'price_error_rows': int(bad_price.sum()),
    'future_end_rows_raw': int(future_raw.sum()),
    'future_end_rows_after_price': int(future_price.sum()),
    'date_error_rows_after_price': int(date_error.sum()),
})
display(check_before)
missing_unknown


raw_rows                       445260
raw_cols                           30
drop_cols                           5
price_error_rows                  572
future_end_rows_raw            430501
future_end_rows_after_price    430497
date_error_rows_after_price      9948
dtype: int64

ads_search         2
ads_summary     7061
ads_save_way     514
ads_guide        303
dtype: int64

## 전처리 적용
불필요 컬럼 제거, Unknown 보정, 비용·날짜 이상치 정리
최종 분석용 광고목록 434,740행 기준

In [14]:
df_master = df_raw.copy()

# 제거 후보 컬럼
drop_cols = ['ads_icon_img', 'ads_sex_type', 'ads_limit', 'ads_package', 'ads_payment']

# Unknown 대체 컬럼
unknown_cols = ['ads_search', 'ads_summary', 'ads_save_way', 'ads_guide']

# 기간 판단 컬럼
date_cols = ['regdate', 'ads_sdate', 'ads_edate']

# 불필요 컬럼 제거
cols_before = df_master.shape[1]
df_master = df_master.drop(columns=drop_cols)
drop_count = cols_before - df_master.shape[1]

# 결측 텍스트 Unknown 대체
df_master[unknown_cols] = df_master[unknown_cols].fillna('Unknown')

# 단가 숫자 변환
df_master['ads_contract_price'] = pd.to_numeric(df_master['ads_contract_price'], errors='coerce')
df_master['ads_reward_price'] = pd.to_numeric(df_master['ads_reward_price'], errors='coerce')

# 비용 정상 조건
contract = df_master['ads_contract_price']
reward = df_master['ads_reward_price']
price_keep = np.logical_and.reduce([
    contract.notna(),
    reward.notna(),
    contract >= reward,
    contract > 0,
    reward > 0,
])

# 비용 오류 제거
rows_before_price = len(df_master)
df_master = df_master.loc[price_keep].reset_index(drop=True)
price_removed = rows_before_price - len(df_master)

# 날짜 변환
for col in date_cols:
    df_master[col] = pd.to_datetime(df_master[col], errors='coerce')

# 미래 종료일 보정
future_end = df_master['ads_edate'].dt.year >= 9999
future_fix_count = int(future_end.sum())
df_master.loc[future_end, 'ads_edate'] = future_fix_date

# 날짜 정상 조건
has_date = df_master[date_cols].notna().all(axis=1)
start_ok = df_master['ads_sdate'] <= analysis_edate
end_ok = df_master['ads_edate'] >= analysis_sdate
date_keep = np.logical_and.reduce([
    has_date,
    start_ok,
    end_ok,
])

# 분석 기간 밖 광고 제거
rows_before_date = len(df_master)
df_master = df_master.loc[date_keep].reset_index(drop=True)
date_removed = rows_before_date - len(df_master)

# 도메인 라벨 연결
df_master = df_master.merge(label_map.drop_duplicates('ads_idx'), on='ads_idx', how='left')

pd.Series({
    'drop_cols': drop_count,
    'price_removed_rows': price_removed,
    'future_end_fixed_rows': future_fix_count,
    'date_error_removed_rows': date_removed,
    'domain_label_added': 1,
    'final_rows': df_master.shape[0],
    'final_cols': df_master.shape[1],
})


drop_cols                       5
price_removed_rows            572
future_end_fixed_rows      430497
date_error_removed_rows      9948
domain_label_added              1
final_rows                 434740
final_cols                     26
dtype: int64

## 저장본 검증
최종 CSV 덮어쓰기 없이 저장본 상태만 확인
행/열, domain_label 결측, ads_idx 중복 점검


In [15]:
if WRITE_FINAL_CSV:
    raise RuntimeError('최종 CSV 덮어쓰기 비활성')

# 저장본 재확인
df_check = pd.read_csv('C:/Users/ckcma/Downloads/temp2/04 ADS/광고 목록_도메인 라벨링_전처리완료.csv', encoding='utf-8-sig')

df_check.shape


(434740, 26)

domain_label 결측과 ads_idx 중복 확인
둘 다 0이면 저장본 기준 이상 없음

In [16]:
pd.Series({
    'domain_label_missing': df_check['domain_label'].isna().sum(),
    'ads_idx_duplicated': df_check['ads_idx'].duplicated().sum(),
})

domain_label_missing    0
ads_idx_duplicated      0
dtype: int64

도메인 라벨 분포 확인
최종 저장본 기준 1~5 라벨 비중 점검

In [17]:
df_check['domain_label'].value_counts().sort_index()

domain_label
1     24235
2      4382
3    141107
4    220496
5     44520
Name: count, dtype: int64

저장본 샘플 확인
광고명과 라벨, 집행 기간 컬럼 연결 상태 점검

In [18]:
df_check[['ads_idx', 'ads_name', 'domain_label', 'ads_sdate', 'ads_edate']].head()

,ads_idx,ads_name,domain_label,ads_sdate,ads_edate
0,160,리니지레드나이츠,1,2016-12-01 00:00:00,2025-08-29
1,284,강철의함대:Ocean Overlord,1,2016-12-29 00:00:00,2025-08-29
2,292,스노우 SNOW,1,2016-12-29 00:00:00,2025-08-29
3,304,서머너즈 워: 천공의 아레나,1,2017-01-01 00:00:00,2025-08-29
4,1166,테스트_설치형,5,2017-01-01 00:00:00,2025-08-29
